In [4]:

!pip install -q language-tool-python textblob nltk pandas gradio openpyxl

In [5]:

import os, re, json
import pandas as pd
import nltk
from textblob import TextBlob
import language_tool_python

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
tool = language_tool_python.LanguageTool('en-US')

In [6]:

def tokenize_words(text):
    return [t for t in nltk.word_tokenize(text) if re.search(r'\w', t)]

In [7]:

def salutation_score(text):
    t = text.lower()

    if "i am excited to introduce" in t or "feeling great" in t:
        return 5, "Excellent salutation"

    if re.search(r"\b(good morning|good afternoon|good evening|good day)\b", t):
        return 4, "Good salutation"

    if re.search(r"\b(hi|hello|hello everyone)\b", t):
        return 2, "Normal greeting"

    return 0, "No salutation"

In [8]:
def keywords_score(text):
    t = text.lower()

    must_have = ["name", "age", "school", "class", "family", "hobby", "interest"]
    good = ["about family", "from", "origin", "ambition", "goal", "dream",
            "interesting", "fun fact", "strength", "achievement"]

    found_must = []

    if re.search(r"\b(my name is|i am|this is|myself)\b", t):
        found_must.append("name")

    for k in must_have:
        if k != "name" and re.search(r"\b" + re.escape(k) + r"\b", t):
            found_must.append(k)

    must_score = min(len(found_must) * 4, 20)

    found_good = [k for k in good if k in t]
    good_score = min(len(found_good) * 2, 10)

    return must_score + good_score, {
        "must_found": found_must,
        "good_found": found_good
    }


In [9]:

def flow_score(text):
    t = text.lower()

    sal = re.search(r"\b(hi|hello|good morning|good afternoon|good evening)\b", t)
    name = re.search(r"\b(my name is|i am|this is|myself)\b", t)
    mandatory = re.search(r"\b(age|class|school|grade|from|family)\b", t)
    optional = re.search(r"\b(hobby|interest|goal|ambition|dream|fun fact|interesting|special thing|i enjoy)\b", t)
    closing = re.search(r"\b(thank you|thanks)\b", t)

    parts = [sal, name, mandatory, optional, closing]

    if any(p is None for p in parts):
        return 0, {"flow": "Missing one or more flow elements"}

    S, N, M, O, C = [p.start() for p in parts]

    if S < N < M < O < C:
        return 5, {"flow": "Correct order"}
    else:
        return 0, {"flow": f"Incorrect order: {S, N, M, O, C}"}

In [10]:

def speech_rate_score(text, duration_sec):
    wc = len(tokenize_words(text))
    if not duration_sec or duration_sec <= 0:
        return 0, {"wpm": None}

    wpm = round(wc / (duration_sec / 60), 2)

    if wpm > 161:
        s = 2
    elif 141 <= wpm <= 160:
        s = 6
    elif 111 <= wpm <= 140:
        s = 10
    elif 81 <= wpm <= 110:
        s = 6
    elif 70 <= wpm <= 80:
        s = 2
    else:
        s = 0

    return s, {"wpm": wpm}

In [11]:
def grammar_score(text):
    matches = tool.check(text)
    errors = len(matches)
    wc = max(1, len(tokenize_words(text)))
    errs_per_100 = errors / (wc / 100)

    frac = max(0, 1 - min(errs_per_100 / 10, 1))

    if frac >= 0.9:
        s = 10
    elif frac >= 0.7:
        s = 8
    elif frac >= 0.5:
        s = 6
    elif frac >= 0.3:
        s = 4
    else:
        s = 2

    return s, {
        "errors": errors,
        "errs_per_100": round(errs_per_100, 3),
        "grammar_fraction": round(frac, 3)
    }


In [12]:
def vocabulary_score(text):
    toks = [t.lower() for t in tokenize_words(text)]

    if not toks:
        return 2, {"ttr": 0}

    ttr = len(set(toks)) / len(toks)

    if ttr >= 0.9:
        s = 10
    elif ttr >= 0.7:
        s = 8
    elif ttr >= 0.5:
        s = 6
    elif ttr >= 0.3:
        s = 4
    else:
        s = 2

    return s, {"ttr": round(ttr, 3)}

In [13]:
def clarity_score(text):
    fillers = ["um","uh","like","you know","so","actually","basically",
               "right","i mean","well","kinda","sort of","okay","hmm","ah"]

    t = text.lower()
    wc = max(1, len(tokenize_words(text)))

    count = 0
    for f in fillers:
        count += len(re.findall(r"\b" + re.escape(f) + r"\b", t))

    rate = (count * 100) / wc

    if rate <= 3:
        s = 15
    elif rate <= 6:
        s = 12
    elif rate <= 9:
        s = 9
    elif rate <= 12:
        s = 6
    else:
        s = 3

    return s, {"fillers_count": count, "filler_rate_percent": round(rate, 2)}


In [14]:
def engagement_score(text):
    p = TextBlob(text).sentiment.polarity
    s_val = (p + 1) / 2

    if s_val >= 0.9:
        s = 15
    elif s_val >= 0.7:
        s = 12
    elif s_val >= 0.5:
        s = 9
    elif s_val >= 0.3:
        s = 6
    else:
        s = 3

    return s, {"polarity": round(p, 3), "scaled": round(s_val, 3)}


In [15]:
def evaluate_exact(text, duration_sec):
    wc = len(tokenize_words(text))

    sal_s, sal_d = salutation_score(text)
    kw_s, kw_d = keywords_score(text)
    flow_s, flow_d = flow_score(text)
    sp_s, sp_d = speech_rate_score(text, duration_sec)
    gr_s, gr_d = grammar_score(text)
    voc_s, voc_d = vocabulary_score(text)
    cla_s, cla_d = clarity_score(text)
    eng_s, eng_d = engagement_score(text)

    results = [
        {"criterion": "Salutation", "weight": 5, "score": sal_s, "details": sal_d},
        {"criterion": "Keywords",   "weight": 30, "score": kw_s, "details": kw_d},
        {"criterion": "Flow",       "weight": 5, "score": flow_s, "details": flow_d},
        {"criterion": "SpeechRate", "weight": 10, "score": sp_s, "details": sp_d},
        {"criterion": "Grammar",    "weight": 10, "score": gr_s, "details": gr_d},
        {"criterion": "Vocabulary", "weight": 10, "score": voc_s, "details": voc_d},
        {"criterion": "Clarity",    "weight": 15, "score": cla_s, "details": cla_d},
        {"criterion": "Engagement", "weight": 15, "score": eng_s, "details": eng_d},
    ]

    total_score = sum(r["score"] for r in results)
    total_weight = 100
    overall = (total_score / total_weight) * 100

    return {
        "words": wc,
        "per_criterion": results,
        "total_score_raw": total_score,
        "total_possible": total_weight,
        "overall_score_0_100": round(overall, 2)
    }


In [ ]:
import gradio as gr

def run_eval(text, duration):
    try:
        return evaluate_exact(text, duration)
    except Exception as e:
        return {"error": str(e)}

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("## 🎤 Student Introduction Scoring Tool")

    with gr.Row():
        transcript = gr.Textbox(
            label="Paste Transcript",
            lines=15,
            placeholder="Enter transcript..."
        )

        output = gr.JSON(label="Scoring Output")

    duration = gr.Number(label="Duration (seconds)", value=52)

    score_btn = gr.Button("Evaluate Score", variant="primary")

    score_btn.click(
        fn=run_eval,
        inputs=[transcript, duration],
        outputs=output
    )

demo.launch(debug=True)

/tmp/ipykernel_496/1203254087.py:9: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://787e44536c3af54e72.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
